In [118]:
import cv2
import os
import time
import json
import numpy as np
from datetime import datetime
from pathlib import Path

BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)

# ====== LOAD CASCADE CLASSIFIERS ======
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)
mouth_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_mcs_mouth.xml'
)

# ====== VERIFY CASCADES LOADED ======
if face_cascade.empty():
    print("⚠ WARNING: Face Cascade failed to load!")
    print(f"Path: {cv2.data.haarcascades}")
else:
    print("✓ Face Cascade loaded")

if mouth_cascade.empty():
    print("⚠ WARNING: Mouth Cascade failed to load!")
else:
    print("✓ Mouth Cascade loaded")

print("✓ Data collection setup ready")

✓ Face Cascade loaded
⚠ WARNING: Mouth Cascade failed to load!
✓ Data collection setup ready


[ERROR:0@3090.221] global persistence.cpp:566 open Can't open file: '/home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/building-model/.venv/lib/python3.11/site-packages/cv2/data/haarcascade_mcs_mouth.xml' in read mode


In [119]:
def extract_mouth_landmarks(frame):
    """
    Extract mouth region & landmarks dari frame dengan error handling
    """
    if frame is None or frame.size == 0:
        return None, None, None
    
    try:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        
        # Detect face
        if face_cascade.empty():
            return None, None, None
            
        faces = face_cascade.detectMultiScale(gray, 1.3, 5, minSize=(30, 30))
        if len(faces) == 0:
            return None, None, None
        
        # Ambil face terbesar
        face_x, face_y, face_w, face_h = max(faces, key=lambda f: f[2] * f[3])
        face_bbox = {"x": int(face_x), "y": int(face_y), "w": int(face_w), "h": int(face_h)}
        
        # Extract mouth region (bottom 40% dari face)
        mouth_y_start = face_y + int(face_h * 0.6)
        mouth_region = gray[mouth_y_start:face_y + face_h, face_x:face_x + face_w]
        
        # Detect mouth
        if mouth_cascade.empty():
            # Fallback: return face bbox dengan simple mouth estimation
            mouth_x = face_x + int(face_w * 0.1)
            mouth_y = mouth_y_start
            mouth_w = int(face_w * 0.8)
            mouth_h = int(face_h * 0.35)
        else:
            mouths = mouth_cascade.detectMultiScale(mouth_region, 1.5, 5, minSize=(10, 10))
            
            if len(mouths) == 0:
                # Fallback: estimate mouth region
                mouth_x = face_x + int(face_w * 0.1)
                mouth_y = mouth_y_start
                mouth_w = int(face_w * 0.8)
                mouth_h = int(face_h * 0.35)
            else:
                # Use detected mouth
                mx, my, mw, mh = max(mouths, key=lambda m: m[2] * m[3])
                mouth_x = face_x + mx
                mouth_y = mouth_y_start + my
                mouth_w = mw
                mouth_h = mh
        
        # Create mouth landmarks (5 points)
        landmarks_points = [
            {"label": "mouth_top_left", "px": mouth_x, "py": mouth_y},
            {"label": "mouth_top_right", "px": mouth_x + mouth_w, "py": mouth_y},
            {"label": "mouth_bottom_left", "px": mouth_x, "py": mouth_y + mouth_h},
            {"label": "mouth_bottom_right", "px": mouth_x + mouth_w, "py": mouth_y + mouth_h},
            {"label": "mouth_center", "px": mouth_x + mouth_w//2, "py": mouth_y + mouth_h//2}
        ]
        
        mouth_landmarks = [
            {
                "label": lp["label"],
                "norm_x": float(lp["px"] / w),
                "norm_y": float(lp["py"] / h),
                "pixel_x": int(lp["px"]),
                "pixel_y": int(lp["py"])
            }
            for lp in landmarks_points
        ]
        
        mouth_bbox = {"x": int(mouth_x), "y": int(mouth_y), "w": int(mouth_w), "h": int(mouth_h)}
        
        return mouth_landmarks, mouth_bbox, face_bbox
        
    except Exception as e:
        print(f"   ⚠ Error detecting landmarks: {e}")
        return None, None, None

def draw_mouth_on_frame(frame, mouth_landmarks, mouth_bbox=None, face_bbox=None):
    """
    Draw mouth landmarks & bounding boxes pada frame
    """
    if mouth_landmarks is None or frame is None:
        return frame
    
    try:
        # Draw mouth landmarks (green circles)
        for landmark in mouth_landmarks:
            x = int(landmark["pixel_x"])
            y = int(landmark["pixel_y"])
            cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)
            cv2.circle(frame, (x, y), 6, (0, 255, 0), 2)
        
        # Draw mouth bounding box (blue rectangle)
        if mouth_bbox is not None:
            x, y, w, h = int(mouth_bbox["x"]), int(mouth_bbox["y"]), int(mouth_bbox["w"]), int(mouth_bbox["h"])
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
            cv2.putText(frame, "MOUTH", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
        
        # Draw face bounding box (cyan rectangle)
        if face_bbox is not None:
            x, y, w, h = int(face_bbox["x"]), int(face_bbox["y"]), int(face_bbox["w"]), int(face_bbox["h"])
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 0), 2)
        
        return frame
    except Exception as e:
        print(f"   ⚠ Error drawing landmarks: {e}")
        return frame

In [120]:
def capture_video_from_camera(output_dir, duration_seconds=10, speaker_name="ada"):
    """
    Capture video dari camera dengan landmark display + error handling
    """
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(0)
    
    # Check if camera opened
    if not cap.isOpened():
        print("❌ ERROR: Cannot open camera!")
        print("   Possible solutions:")
        print("   1. Check if camera is connected")
        print("   2. Check camera permissions: sudo chown $USER /dev/video*")
        print("   3. Try different camera index: VideoCapture(1), VideoCapture(2), etc.")
        return None, 0, 0
    
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Validate resolution
    if width == 0 or height == 0:
        print(f"❌ ERROR: Invalid camera resolution {width}x{height}")
        print(f"   Try: cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)")
        cap.release()
        return None, 0, 0
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(output_dir, f"{speaker_name}_{timestamp}.avi")
    
    # ====== USE MJPEG CODEC (more compatible) ======
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # ====== CHECK IF VIDEO WRITER OPENED ======
    if not out.isOpened():
        print(f"❌ ERROR: Cannot create video writer with MJPG!")
        print(f"   Trying fallback codec...")
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        if not out.isOpened():
            cap.release()
            print(f"❌ FATAL: Video writer failed with both MJPG and XVID codecs!")
            return None, 0, 0
    
    print(f"\n🎥 Recording untuk {duration_seconds} detik...")
    print(f"   Resolution: {width}x{height}, FPS: {fps}")
    print("   Press 'q' untuk stop lebih awal")
    start_time = time.time()
    frame_count = 0
    landmark_count = 0
    write_error_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("⚠ Cannot read frame from camera")
            break
        
        # ====== DETECT LANDMARKS SAAT RECORDING ======
        mouth_landmarks, mouth_bbox, face_bbox = extract_mouth_landmarks(frame)
        if mouth_landmarks is not None:
            frame = draw_mouth_on_frame(frame, mouth_landmarks, mouth_bbox, face_bbox)
            landmark_count += 1
        
        # ====== WRITE FRAME WITH ERROR CHECK ======
        try:
            out.write(frame)
        except Exception as e:
            write_error_count += 1
            print(f"⚠ Error writing frame: {e}")
            if write_error_count > 5:
                print("❌ Too many write errors, stopping recording")
                break
        
        elapsed = int(time.time() - start_time)
        status = "✓ DETECTED" if mouth_landmarks is not None else "⚠ No mouth"
        cv2.putText(frame, f"Recording... {frame_count} | {elapsed}s | {status}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow('Recording', frame)
        
        frame_count += 1
        
        if time.time() - start_time > duration_seconds:
            break
            
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("   → Recording dihentikan oleh user")
            break
    
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    
    # ====== VERIFY VIDEO FILE WAS CREATED ======
    if os.path.exists(output_path):
        file_size = os.path.getsize(output_path)
        print(f"✓ Video saved: {output_path} ({file_size} bytes)")
    else:
        print(f"❌ ERROR: Video file was not created!")
        return None, 0, 0
    
    print(f"✓ Total frames recorded: {frame_count}")
    
    # ====== FIX: SAFE DIVISION ======
    if frame_count > 0:
        detection_rate = 100 * landmark_count / frame_count
        print(f"✓ Landmarks detected: {landmark_count}/{frame_count} ({detection_rate:.1f}%)")
    else:
        print(f"⚠ WARNING: No frames recorded!")
        print(f"   Camera might not be accessible or recording failed")
        landmark_count = 0
    
    return output_path, frame_count, fps

In [121]:
def extract_frames_from_video(video_path, output_frames_dir, skip_frames=1):
    """
    Extract frames dari video + detect mouth landmarks
    Returns: (frame_paths, extracted_count, landmarks_data)
    """
    os.makedirs(output_frames_dir, exist_ok=True)
    
    # ====== CHECK IF VIDEO FILE EXISTS ======
    if not os.path.exists(video_path):
        print(f"\n❌ Video file not found: {video_path}")
        return [], 0, []
    else:
        file_size = os.path.getsize(video_path)
        print(f"\n📸 Video file info: {file_size} bytes")
    
    cap = cv2.VideoCapture(video_path)
    
    # ====== CHECK IF VIDEO OPENED SUCCESSFULLY ======
    if not cap.isOpened():
        print(f"\n❌ Cannot open video file: {video_path}")
        print(f"   Possible causes:")
        print(f"   1. Video file is corrupted (size: {os.path.getsize(video_path)} bytes)")
        print(f"   2. Video codec not supported (try using .avi or .mov)")
        print(f"   3. File path has special characters")
        return [], 0, []
    
    # Get video properties
    frame_count_prop = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_prop = cap.get(cv2.CAP_PROP_FPS)
    
    print(f"📸 Extracting frames dari video...")
    print(f"   + Video: {video_path}")
    print(f"   + Total frames in video: {frame_count_prop}")
    print(f"   + FPS: {fps_prop}")
    print(f"   + Skip frames: {skip_frames}")
    print(f"   + Detecting mouth landmarks...")
    
    frame_count = 0
    extracted_count = 0
    frame_paths = []
    landmarks_data = []
    
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Simpan setiap N frame
        if frame_count % skip_frames == 0:
            frame_name = f"frame_{extracted_count:04d}.jpg"
            frame_path = os.path.join(output_frames_dir, frame_name)
            
            # ====== DETECT LANDMARKS ======
            mouth_landmarks, mouth_bbox, face_bbox = extract_mouth_landmarks(frame)
            
            # DEBUG: Print detection status
            if extracted_count < 3 or extracted_count % 20 == 0:
                status = "✓ DETECTED" if mouth_landmarks is not None else "⚠ NO_MOUTH"
                print(f"   Frame {extracted_count}: {status}")
            
            # Draw landmarks pada frame
            if mouth_landmarks is not None:
                frame = draw_mouth_on_frame(frame, mouth_landmarks, mouth_bbox, face_bbox)
            
            # Save frame dengan landmarks
            try:
                cv2.imwrite(frame_path, frame)
            except Exception as e:
                print(f"   ⚠ Error saving frame {extracted_count}: {e}")
                continue
                
            frame_paths.append(frame_path)
            
            # Store landmarks info
            landmark_entry = {
                "frame_num": extracted_count,
                "frame_name": frame_name,
                "detected": mouth_landmarks is not None
            }
            
            # Simpan landmark details jika detected
            if mouth_landmarks is not None:
                landmark_entry["mouth_landmarks"] = mouth_landmarks
                landmark_entry["mouth_bbox"] = mouth_bbox
                landmark_entry["face_bbox"] = face_bbox
            else:
                landmark_entry["mouth_landmarks"] = None
                landmark_entry["mouth_bbox"] = None
                landmark_entry["face_bbox"] = None
            
            landmarks_data.append(landmark_entry)
            extracted_count += 1
        
        frame_count += 1
    
    cap.release()
    
    # ====== DETAILED RESULTS ======
    if extracted_count > 0:
        detected_count = sum(1 for l in landmarks_data if l["detected"])
        detection_rate = 100 * detected_count / extracted_count
        print(f"\n✅ Extracted {extracted_count} frames dari {frame_count} total frames")
        print(f"✓ Mouth landmarks detected di {detected_count}/{extracted_count} frames ({detection_rate:.1f}%)")
        print(f"✓ Frames saved to: {output_frames_dir}")
    else:
        if frame_count == 0:
            print(f"\n⚠ No frames found in video!")
            print(f"   Video file size: {os.path.getsize(video_path)} bytes")
            print(f"   Video may be corrupted or empty")
        else:
            print(f"\n⚠ Frames in video: {frame_count}, but skip_frames={skip_frames}")
            print(f"   Try reducing skip_frames (currently: {skip_frames})")
            print(f"   Example: extract_frames_from_video(video_path, output_dir, skip_frames=1)")
    
    return frame_paths, extracted_count, landmarks_data

In [122]:
def collect_lip_reading_data(kata=None, speaker_name=None, duration=20, skip_frames=1):
    """
    Collect lip reading data dengan mouth landmarks
    """
    
    if kata is None:
        kata = input("\n📝 Masukkan KATA yang akan diucapkan (contoh: halo, terima_kasih, apa_kabar): ").strip().lower()
        if not kata:
            print("❌ Kata tidak boleh kosong!")
            return None
    
    if speaker_name is None:
        speaker_name = input("👤 Masukkan nama SPEAKER (contoh: ada, budi, citra): ").strip().lower()
        if not speaker_name:
            print("❌ Nama speaker tidak boleh kosong!")
            return None
    
    print(f"\n{'='*60}")
    print(f"📝 KATA          : {kata.upper()}")
    print(f"👤 SPEAKER       : {speaker_name.upper()}")
    print(f"{'='*60}")
    
    # Setup directories
    kata_dir = os.path.join(BASE_DIR, kata)
    speaker_dir = os.path.join(kata_dir, f"speaker_{speaker_name}")
    videos_dir = os.path.join(speaker_dir, "raw", "videos")
    frames_dir = os.path.join(speaker_dir, "raw", "frames")
    
    os.makedirs(videos_dir, exist_ok=True)
    os.makedirs(frames_dir, exist_ok=True)
    
    # 1. Record video
    print(f"\n💬 Bersiaplah meucapkan kata: '{kata.upper()}'")
    result = capture_video_from_camera(
        videos_dir, 
        duration, 
        f"{kata}_{speaker_name}"
    )
    
    # ====== CHECK CAMERA RECORDING RESULT ======
    if result is None or result[1] == 0:
        print(f"\n❌ Recording failed! No frames captured.")
        return None
    
    video_path, frame_count, fps = result
    
    # 2. Extract frames + detect landmarks
    frame_paths, extracted_count, landmarks_data = extract_frames_from_video(
        video_path, 
        frames_dir, 
        skip_frames
    )
    
    # ====== CHECK IF FRAMES WERE EXTRACTED ======
    if extracted_count == 0:
        print(f"\n❌ No frames were extracted!")
        print(f"   Please check skip_frames parameter or video file")
        return None
    
    # 3. Build metadata dengan landmarks
    frames_metadata = []
    for i in range(len(frame_paths)):
        frame_entry = {
            "frame_num": i,
            "frame_name": landmarks_data[i]["frame_name"],
            "path": frame_paths[i],
            "detected": landmarks_data[i]["detected"]
        }
        
        # Tambah landmark details jika ada
        if landmarks_data[i]["detected"]:
            frame_entry["mouth_landmarks"] = landmarks_data[i]["mouth_landmarks"]
            frame_entry["mouth_bbox"] = landmarks_data[i]["mouth_bbox"]
            frame_entry["face_bbox"] = landmarks_data[i]["face_bbox"]
        
        frames_metadata.append(frame_entry)
    
    # 4. Save metadata
    metadata = {
        "kata": kata,
        "speaker": speaker_name,
        "timestamp": datetime.now().isoformat(),
        "video_path": video_path,
        "frames_dir": frames_dir,
        "total_frames_recorded": frame_count,
        "total_frames_extracted": extracted_count,
        "fps": fps,
        "skip_frames": skip_frames,
        "detector": "OpenCV Cascade Classifier",
        "frames": frames_metadata
    }
    
    metadata_path = os.path.join(speaker_dir, "metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"\n✓ Metadata saved: {metadata_path}")
    
    # Summary
    detected_total = sum(1 for f in frames_metadata if f["detected"])
    
    # ====== SAFE DIVISION ======
    if detected_total > 0 and extracted_count > 0:
        detection_rate = 100 * detected_total / extracted_count
    else:
        detection_rate = 0
    
    print(f"\n{'='*60}")
    print(f"📊 DATA COLLECTION SUMMARY")
    print(f"{'='*60}")
    print(f"📝 KATA                : {kata.upper()}")
    print(f"👤 SPEAKER             : {speaker_name.upper()}")
    print(f"🎥 Video path          : {video_path}")
    print(f"📸 Frames path         : {frames_dir}")
    print(f"✓ Total frames         : {extracted_count}")
    print(f"🎯 Landmarks detected  : {detected_total}/{extracted_count} ({detection_rate:.1f}%)")
    print(f"📋 Metadata path       : {metadata_path}")
    print(f"{'='*60}\n")
    
    return metadata

In [123]:
# Jalankan data collection
metadata = collect_lip_reading_data(duration=20, skip_frames=1)


📝 KATA          : KEREN
👤 SPEAKER       : FAHRI

💬 Bersiaplah meucapkan kata: 'KEREN'

🎥 Recording untuk 20 detik...
   Resolution: 640x480, FPS: 30
   Press 'q' untuk stop lebih awal
✓ Video saved: datasets/keren/speaker_fahri/raw/videos/keren_fahri_20260417_220450.avi (7082696 bytes)
✓ Total frames recorded: 193
✓ Landmarks detected: 111/193 (57.5%)

📸 Video file info: 7082696 bytes
📸 Extracting frames dari video...
   + Video: datasets/keren/speaker_fahri/raw/videos/keren_fahri_20260417_220450.avi
   + Total frames in video: 193
   + FPS: 30.0
   + Skip frames: 1
   + Detecting mouth landmarks...
   Frame 0: ⚠ NO_MOUTH
   Frame 1: ✓ DETECTED
   Frame 2: ⚠ NO_MOUTH
   Frame 20: ⚠ NO_MOUTH
   Frame 40: ⚠ NO_MOUTH
   Frame 60: ⚠ NO_MOUTH
   Frame 80: ⚠ NO_MOUTH
   Frame 100: ⚠ NO_MOUTH
   Frame 120: ⚠ NO_MOUTH
   Frame 140: ⚠ NO_MOUTH
   Frame 160: ⚠ NO_MOUTH
   Frame 180: ⚠ NO_MOUTH

✅ Extracted 193 frames dari 193 total frames
✓ Mouth landmarks detected di 38/193 frames (19.7%)
✓ Fr

In [124]:
# ====== DATA VISUALIZATION & ANALYSIS ======
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

def visualize_collected_data(kata, speaker_name=None):
    """
    Visualize semua collected data untuk kata tertentu
    """
    kata_dir = os.path.join(BASE_DIR, kata.lower())
    
    if not os.path.exists(kata_dir):
        print(f"❌ No data found for kata: {kata}")
        return
    
    # Get all speakers
    speakers = [d for d in os.listdir(kata_dir) if d.startswith("speaker_")]
    
    if not speakers:
        print(f"❌ No speakers found for kata: {kata}")
        return
    
    if speaker_name:
        speakers = [f"speaker_{speaker_name}"] if f"speaker_{speaker_name}" in speakers else []
    
    # Create visualization per speaker
    for speaker_dir_name in speakers:
        speaker_path = os.path.join(kata_dir, speaker_dir_name)
        frames_dir = os.path.join(speaker_path, "raw", "frames")
        metadata_path = os.path.join(speaker_path, "metadata.json")
        
        if not os.path.exists(frames_dir) or not os.path.exists(metadata_path):
            continue
        
        # Load metadata
        with open(metadata_path) as f:
            metadata = json.load(f)
        
        frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
        
        if not frame_files:
            continue
        
        # Create grid untuk show samples
        n_frames = min(6, len(frame_files))
        fig = plt.figure(figsize=(15, 4))
        gs = GridSpec(2, n_frames, figure=fig, hspace=0.3, wspace=0.3)
        
        speaker_name_clean = speaker_dir_name.replace("speaker_", "")
        fig.suptitle(f"Kata: '{kata.upper()}' | Speaker: {speaker_name_clean.upper()}", 
                     fontsize=14, fontweight='bold')
        
        # Sample setiap N frame
        sample_indices = np.linspace(0, len(frame_files)-1, n_frames, dtype=int)
        
        for idx, frame_idx in enumerate(sample_indices):
            frame_path = os.path.join(frames_dir, frame_files[frame_idx])
            frame = cv2.imread(frame_path)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Raw frame
            ax = fig.add_subplot(gs[0, idx])
            ax.imshow(frame)
            ax.set_title(f"Frame {frame_idx}", fontsize=10)
            ax.axis('off')
            
            # Mouth region dengan bounding box
            ax = fig.add_subplot(gs[1, idx])
            ax.imshow(frame)
            
            # Draw mouth bbox dari metadata
            if frame_idx < len(metadata['frames']):
                frame_meta = metadata['frames'][frame_idx]
                if frame_meta['detected'] and frame_meta['mouth_bbox']:
                    bbox = frame_meta['mouth_bbox']
                    rect = patches.Rectangle(
                        (bbox['x'], bbox['y']), bbox['w'], bbox['h'],
                        linewidth=2, edgecolor='lime', facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.set_title(f"Mouth Detected ✓", fontsize=10, color='green')
                else:
                    ax.set_title(f"No Detection ⚠", fontsize=10, color='red')
            
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print(f"\n{'='*60}")
        print(f"📊 DATA STATISTICS: {speaker_name_clean.upper()}")
        print(f"{'='*60}")
        print(f"Kata: {metadata['kata']}")
        print(f"Speaker: {metadata['speaker']}")
        print(f"Timestamp: {metadata['timestamp']}")
        print(f"Total frames recorded: {metadata['total_frames_recorded']}")
        print(f"Total frames extracted: {metadata['total_frames_extracted']}")
        print(f"Skip frames: {metadata['skip_frames']}")
        print(f"FPS: {metadata['fps']}")
        
        # Detection statistics
        detected = sum(1 for f in metadata['frames'] if f['detected'])
        detection_rate = 100 * detected / len(metadata['frames'])
        print(f"Mouth landmarks detected: {detected}/{len(metadata['frames'])} ({detection_rate:.1f}%)")
        print(f"Video file: {metadata['video_path']}")
        print(f"Frame directory: {metadata['frames_dir']}")
        print(f"{'='*60}\n")

print("✓ Visualization functions ready")


✓ Visualization functions ready


In [125]:
def get_dataset_statistics():
    """
    Get overall statistics dari semua collected data
    """
    if not os.path.exists(BASE_DIR):
        print("❌ Dataset directory not found")
        return
    
    total_samples = 0
    total_frames = 0
    total_detected_frames = 0
    words_dict = {}
    speakers_dict = {}
    
    print("\n📊 SCANNING DATASET...\n")
    
    # Scan all words
    for word_dir in os.listdir(BASE_DIR):
        word_path = os.path.join(BASE_DIR, word_dir)
        if not os.path.isdir(word_path):
            continue
        
        # Scan all speakers
        for speaker_dir in os.listdir(word_path):
            speaker_path = os.path.join(word_path, speaker_dir)
            metadata_path = os.path.join(speaker_path, "metadata.json")
            
            if not os.path.exists(metadata_path):
                continue
            
            try:
                with open(metadata_path) as f:
                    metadata = json.load(f)
                
                # Update statistics
                total_samples += 1
                word = metadata['kata'].lower()
                speaker = metadata['speaker'].lower()
                frames_extracted = metadata['total_frames_extracted']
                detected_frames = sum(1 for f in metadata['frames'] if f['detected'])
                
                total_frames += frames_extracted
                total_detected_frames += detected_frames
                
                # Word statistics
                if word not in words_dict:
                    words_dict[word] = {"count": 0, "frames": 0, "detected": 0}
                words_dict[word]['count'] += 1
                words_dict[word]['frames'] += frames_extracted
                words_dict[word]['detected'] += detected_frames
                
                # Speaker statistics
                if speaker not in speakers_dict:
                    speakers_dict[speaker] = {"count": 0, "frames": 0}
                speakers_dict[speaker]['count'] += 1
                speakers_dict[speaker]['frames'] += frames_extracted
                
            except Exception as e:
                print(f"⚠ Error reading {metadata_path}: {e}")
    
    # Print overview
    print(f"{'='*70}")
    print(f"📈 DATASET OVERVIEW")
    print(f"{'='*70}")
    print(f"✓ Total samples: {total_samples}")
    print(f"✓ Total frames: {total_frames}")
    print(f"✓ Detected frames: {total_detected_frames}")
    
    if total_frames > 0:
        detection_rate = 100 * total_detected_frames / total_frames
        print(f"✓ Overall detection rate: {detection_rate:.1f}%")
    
    print(f"\n{'='*70}")
    print(f"📝 WORDS STATISTICS")
    print(f"{'='*70}")
    for word in sorted(words_dict.keys()):
        stats = words_dict[word]
        det_rate = 100 * stats['detected'] / stats['frames'] if stats['frames'] > 0 else 0
        print(f"{word.upper():20} | Samples: {stats['count']:2} | Frames: {stats['frames']:4} | Detection: {det_rate:5.1f}%")
    
    print(f"\n{'='*70}")
    print(f"👤 SPEAKERS STATISTICS")
    print(f"{'='*70}")
    for speaker in sorted(speakers_dict.keys()):
        stats = speakers_dict[speaker]
        print(f"{speaker.upper():20} | Samples: {stats['count']:2} | Total frames: {stats['frames']:4}")
    
    print(f"\n{'='*70}\n")
    
    return {
        "total_samples": total_samples,
        "total_frames": total_frames,
        "total_detected": total_detected_frames,
        "words": words_dict,
        "speakers": speakers_dict
    }

print("✓ Dataset statistics function ready")


✓ Dataset statistics function ready


In [126]:
def batch_collect_data(collection_config):
    """
    Batch data collection dengan config
    
    collection_config format:
    [
        {"kata": "halo", "speaker": "fahri", "count": 3},
        {"kata": "halo", "speaker": "budi", "count": 2},
        {"kata": "terima_kasih", "speaker": "fahri", "count": 3}
    ]
    """
    print(f"\n{'='*70}")
    print(f"🚀 BATCH DATA COLLECTION")
    print(f"{'='*70}\n")
    
    total_to_collect = sum(item['count'] for item in collection_config)
    results = []
    
    for idx, config in enumerate(collection_config):
        kata = config['kata'].lower()
        speaker = config['speaker'].lower()
        count = config['count']
        
        print(f"\n📍 [{idx+1}/{len(collection_config)}] Kata: {kata.upper()} | Speaker: {speaker.upper()} | Samples: {count}")
        print(f"{'-'*70}")
        
        for sample_num in range(count):
            print(f"\n  Sample {sample_num+1}/{count}")
            
            try:
                metadata = collect_lip_reading_data(
                    kata=kata,
                    speaker_name=speaker,
                    duration=20,
                    skip_frames=1
                )
                
                if metadata:
                    results.append({
                        "kata": kata,
                        "speaker": speaker,
                        "status": "✓ Success",
                        "frames": metadata['total_frames_extracted'],
                        "detected": sum(1 for f in metadata['frames'] if f['detected'])
                    })
                else:
                    results.append({
                        "kata": kata,
                        "speaker": speaker,
                        "status": "❌ Failed",
                        "frames": 0,
                        "detected": 0
                    })
                    
            except KeyboardInterrupt:
                print("\n⚠️  Collection interrupted by user")
                return results
            except Exception as e:
                print(f"  ❌ Error: {e}")
                results.append({
                    "kata": kata,
                    "speaker": speaker,
                    "status": f"❌ Error: {str(e)[:30]}",
                    "frames": 0,
                    "detected": 0
                })
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"📊 BATCH COLLECTION SUMMARY")
    print(f"{'='*70}")
    
    successful = sum(1 for r in results if r['status'].startswith('✓'))
    failed = len(results) - successful
    
    print(f"Total collected: {len(results)}/{total_to_collect}")
    print(f"Successful: {successful} ✓")
    print(f"Failed: {failed} ❌")
    print(f"\nDetails:")
    
    for result in results:
        status_icon = "✓" if result['status'].startswith('✓') else "❌"
        print(f"  {status_icon} {result['kata'].upper():20} | {result['speaker'].upper():15} | "
              f"Frames: {result['frames']:3} | Detected: {result['detected']:3} | {result['status']}")
    
    print(f"{'='*70}\n")
    
    return results

print("✓ Batch data collection function ready")


✓ Batch data collection function ready


In [127]:
def check_data_quality():
    """
    Check data quality dan identify issues
    """
    print(f"\n{'='*70}")
    print(f"🔍 DATA QUALITY CHECK")
    print(f"{'='*70}\n")
    
    issues = []
    warnings = []
    
    if not os.path.exists(BASE_DIR):
        print("❌ Dataset directory not found")
        return
    
    # Check each dataset
    for word_dir in os.listdir(BASE_DIR):
        word_path = os.path.join(BASE_DIR, word_dir)
        if not os.path.isdir(word_path):
            continue
        
        for speaker_dir in os.listdir(word_path):
            speaker_path = os.path.join(word_path, speaker_dir)
            metadata_path = os.path.join(speaker_path, "metadata.json")
            frames_dir = os.path.join(speaker_path, "raw", "frames")
            videos_dir = os.path.join(speaker_path, "raw", "videos")
            
            dataset_name = f"{word_dir}/{speaker_dir}"
            
            # 1. Check metadata exists
            if not os.path.exists(metadata_path):
                issues.append(f"❌ Missing metadata.json: {dataset_name}")
                continue
            
            try:
                with open(metadata_path) as f:
                    metadata = json.load(f)
                
                # 2. Check frames directory exists
                if not os.path.exists(frames_dir):
                    issues.append(f"❌ Missing frames directory: {dataset_name}")
                    continue
                
                frame_files = [f for f in os.listdir(frames_dir) if f.endswith('.jpg')]
                extracted = len(frame_files)
                expected = metadata.get('total_frames_extracted', 0)
                
                # 3. Check frame count match
                if extracted != expected:
                    warnings.append(
                        f"⚠  Frame count mismatch in {dataset_name}: "
                        f"metadata says {expected}, but found {extracted}"
                    )
                
                # 4. Check detection rate
                detected = sum(1 for f in metadata.get('frames', []) if f.get('detected'))
                if extracted > 0:
                    det_rate = detected / extracted
                    if det_rate < 0.5:
                        warnings.append(
                            f"⚠  Low detection rate in {dataset_name}: {det_rate:.1%}"
                        )
                
                # 5. Check frame quality (file sizes)
                frame_sizes = []
                for frame_file in frame_files:
                    frame_path = os.path.join(frames_dir, frame_file)
                    frame_sizes.append(os.path.getsize(frame_path))
                
                if frame_sizes:
                    avg_size = np.mean(frame_sizes)
                    if avg_size < 5000:  # Less than 5KB
                        warnings.append(
                            f"⚠  Small frame files in {dataset_name}: avg {avg_size:.0f} bytes"
                        )
                
                # 6. Check video file
                video_files = os.listdir(videos_dir) if os.path.exists(videos_dir) else []
                if not video_files:
                    warnings.append(f"⚠  No video file found in {dataset_name}")
                    
            except Exception as e:
                issues.append(f"❌ Error reading {dataset_name}: {str(e)[:50]}")
    
    # Print results
    print(f"📋 RESULTS:")
    print(f"  Issues found: {len(issues)}")
    print(f"  Warnings: {len(warnings)}\n")
    
    if issues:
        print(f"🔴 ISSUES (Critical):")
        for issue in issues:
            print(f"   {issue}")
    
    if warnings:
        print(f"\n🟡 WARNINGS:")
        for warning in warnings:
            print(f"   {warning}")
    
    if not issues and not warnings:
        print("✅ All checks passed!")
    
    print(f"\n{'='*70}\n")

def cleanup_bad_samples(word=None, speaker=None, min_detection_rate=0.3):
    """
    Remove samples dengan detection rate rendah
    """
    print(f"\n{'='*70}")
    print(f"🧹 CLEANUP - Removing low quality samples")
    print(f"{'='*70}\n")
    print(f"Min detection rate: {min_detection_rate:.1%}\n")
    
    removed_count = 0
    
    for word_dir in os.listdir(BASE_DIR):
        if word and word.lower() != word_dir.lower():
            continue
        
        word_path = os.path.join(BASE_DIR, word_dir)
        if not os.path.isdir(word_path):
            continue
        
        for speaker_dir in os.listdir(word_path):
            if speaker and f"speaker_{speaker.lower()}" != speaker_dir.lower():
                continue
            
            speaker_path = os.path.join(word_path, speaker_dir)
            metadata_path = os.path.join(speaker_path, "metadata.json")
            
            if not os.path.exists(metadata_path):
                continue
            
            try:
                with open(metadata_path) as f:
                    metadata = json.load(f)
                
                detected = sum(1 for f in metadata.get('frames', []) if f.get('detected'))
                total = len(metadata.get('frames', []))
                
                if total > 0:
                    det_rate = detected / total
                    if det_rate < min_detection_rate:
                        # Remove this sample
                        import shutil
                        print(f"Removing {word_dir}/{speaker_dir} (detection rate: {det_rate:.1%})")
                        shutil.rmtree(speaker_path)
                        removed_count += 1
                        
            except Exception as e:
                print(f"⚠  Error processing {word_dir}/{speaker_dir}: {e}")
    
    print(f"\n✓ Removed {removed_count} low quality samples")
    print(f"{'='*70}\n")

print("✓ Data quality functions ready")


✓ Data quality functions ready


# 📊 LIP READING DATA COLLECTION GUIDE

## 🎯 Quick Start

### Single Sample Collection
```python
# Collect single video sample
metadata = collect_lip_reading_data(kata='halo', speaker_name='fahri', duration=10, skip_frames=1)
```

### Batch Collection
```python
# Define collection plan
config = [
    {"kata": "halo", "speaker": "fahri", "count": 3},
    {"kata": "halo", "speaker": "budi", "count": 3},
    {"kata": "terima_kasih", "speaker": "fahri", "count": 3},
]

# Collect all
results = batch_collect_data(config)
```

## 📈 Monitoring & Analysis

### Check Dataset Statistics
```python
stats = get_dataset_statistics()
```

### Visualize Collected Data
```python
# View single speaker
visualize_collected_data(kata='halo', speaker_name='fahri')

# View all speakers for a word
visualize_collected_data(kata='halo')
```

### Data Quality Check
```python
# Find issues
check_data_quality()

# Cleanup low quality samples
cleanup_bad_samples(min_detection_rate=0.5)
```

## 📁 Dataset Structure
```
datasets/
├── halo/
│   ├── speaker_fahri/
│   │   └── raw/
│   │       ├── frames/
│   │       │   ├── frame_0000.jpg
│   │       │   └── ...
│   │       └── videos/
│   │           └── halo_fahri_YYYYMMDD_HHMMSS.avi
│   │   └── metadata.json
│   └── speaker_budi/
├── terima_kasih/
│   └── speaker_fahri/
└── ...
```

## 🎥 Data Collection Tips

### Video Quality
- ✅ **Lighting**: Good natural light or consistent artificial lighting
- ✅ **Distance**: 30-60cm from camera
- ✅ **Position**: Face centered, slight angle variation okay
- ✅ **Background**: Plain background preferred
- ❌ **Avoid**: Backlighting, side lighting that creates shadows

### Recording Tips
- Speak naturally and clearly
- Vary speed slightly (normal, slow, fast)
- Multiple attempts per word reduces overfitting
- Press 'q' to stop recording early if needed

### Optimal Settings
| Parameter | Value | Notes |
|-----------|-------|-------|
| Duration | 10 sec | 2-3 words per sample |
| FPS | 30 | Standard video framerate |
| Skip Frames | 1 | Extract every frame (30 FPS) |
| Resolution | 640x480+ | Minimum for mouth detail |
| Mouth Detection | >70% | Acceptable threshold |

## 🔍 Quality Metrics

### What to Check
1. **Detection Rate**: % of frames with mouth detected
   - Good: >80%
   - Acceptable: 50-80%
   - Poor: <50%

2. **Frame Count**: Frames extracted per sample
   - Target: 15-30 frames per sample
   - Less: May be too fast or bad lighting
   - More: May be redundant

3. **File Sizes**: Individual frame sizes
   - Normal: 10-30 KB per frame
   - Too small: May indicate compression issues
   - Too large: May need optimization

## 🚨 Troubleshooting

### No Frames Extracted
- Increase `duration` parameter
- Reduce `skip_frames` value
- Check camera is working

### Low Detection Rate
- Improve lighting
- Move closer to camera
- Adjust head position
- Try cascade classifier parameters

### Camera Issues
- Check permissions: `sudo chown $USER /dev/video*`
- Try different camera index in `capture_video_from_camera`
- Check if camera is in use by another app

In [128]:
# ====== PRACTICAL EXAMPLE: COMPLETE DATA COLLECTION WORKFLOW ======

# Example 1: Single sample
print("="*70)
print("EXAMPLE 1: Single Sample Collection")
print("="*70)
print("""
# Collect one video
metadata = collect_lip_reading_data(
    kata='halo',
    speaker_name='fahri', 
    duration=20,
    skip_frames=1
)

# View the sample
visualize_collected_data(kata='halo', speaker_name='fahri')
""")

print("\n" + "="*70)
print("EXAMPLE 2: Batch Collection (Recommended)")
print("="*70)
print("""
# Plan your data collection
collection_plan = [
    # Kata 1: HALO
    {"kata": "halo", "speaker": "fahri", "count": 5},
    {"kata": "halo", "speaker": "budi", "count": 3},
    {"kata": "halo", "speaker": "citra", "count": 3},
    
    # Kata 2: TERIMA KASIH
    {"kata": "terima_kasih", "speaker": "fahri", "count": 5},
    {"kata": "terima_kasih", "speaker": "budi", "count": 3},
    
    # Kata 3: APA KABAR
    {"kata": "apa_kabar", "speaker": "fahri", "count": 5},
]

# Collect all data
results = batch_collect_data(collection_plan)

# Check results
stats = get_dataset_statistics()

# Visualize
visualize_collected_data('halo')
visualize_collected_data('terima_kasih')
visualize_collected_data('apa_kabar')
""")

print("\n" + "="*70)
print("EXAMPLE 3: Data Quality Workflow")
print("="*70)
print("""
# 1. Check for issues
check_data_quality()

# 2. Remove low quality samples (< 50% detection rate)
cleanup_bad_samples(min_detection_rate=0.5)

# 3. Verify cleaned data
stats = get_dataset_statistics()
check_data_quality()

# 4. Visualize final dataset
for word in ['halo', 'terima_kasih', 'apa_kabar']:
    visualize_collected_data(word)
""")

print("\n" + "="*70)
print("✅ DATA COLLECTION READY!")
print("="*70)
print("""
🎬 NEXT STEPS:
1. Run collect_lip_reading_data() or batch_collect_data() to collect
2. Use get_dataset_statistics() to monitor progress
3. Use visualize_collected_data() to verify quality
4. Use check_data_quality() to find issues
5. Use cleanup_bad_samples() to improve dataset

💡 TIP: Start small (1-2 words, 2-3 speakers) before scaling up!
""")

EXAMPLE 1: Single Sample Collection

# Collect one video
metadata = collect_lip_reading_data(
    kata='halo',
    speaker_name='fahri', 
    duration=20,
    skip_frames=1
)

# View the sample
visualize_collected_data(kata='halo', speaker_name='fahri')


EXAMPLE 2: Batch Collection (Recommended)

# Plan your data collection
collection_plan = [
    # Kata 1: HALO
    {"kata": "halo", "speaker": "fahri", "count": 5},
    {"kata": "halo", "speaker": "budi", "count": 3},
    {"kata": "halo", "speaker": "citra", "count": 3},

    # Kata 2: TERIMA KASIH
    {"kata": "terima_kasih", "speaker": "fahri", "count": 5},
    {"kata": "terima_kasih", "speaker": "budi", "count": 3},

    # Kata 3: APA KABAR
    {"kata": "apa_kabar", "speaker": "fahri", "count": 5},
]

# Collect all data
results = batch_collect_data(collection_plan)

# Check results
stats = get_dataset_statistics()

# Visualize
visualize_collected_data('halo')
visualize_collected_data('terima_kasih')
visualize_collected_data('apa_kab

In [129]:
# ====== DIAGNOSTIC: CEK KAMERA & FPS ======
def diagnose_camera_settings():
    """
    Diagnosa setting kamera & FPS yang sebenarnya
    """
    print(f"\n{'='*70}")
    print(f"🔍 CAMERA DIAGNOSTIC")
    print(f"{'='*70}\n")
    
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ Kamera tidak bisa dibuka!")
        return
    
    # Get properties
    fps_prop = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"📹 Kamera Properties (dari driver):")
    print(f"   - FPS: {fps_prop}")
    print(f"   - Resolution: {width}x{height}")
    print(f"   - Frame count property: {frame_count}")
    
    # Test actual FPS dengan merekam 5 detik
    print(f"\n⏱️  Testing actual FPS (recording 5 seconds)...")
    
    start_time = time.time()
    frame_count = 0
    
    while time.time() - start_time < 5:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
    
    actual_time = time.time() - start_time
    actual_fps = frame_count / actual_time
    
    print(f"   ✓ Frames captured: {frame_count}")
    print(f"   ✓ Time elapsed: {actual_time:.2f}s")
    print(f"   ✓ Actual FPS: {actual_fps:.2f}")
    
    cap.release()
    
    # Analysis
    print(f"\n📊 ANALISIS:")
    if actual_fps < 15:
        print(f"   ⚠️  FPS rendah! Hanya {actual_fps:.1f}fps")
        print(f"      Untuk 10 detik: {int(actual_fps * 10)} frame (seharusnya ~300 frame)")
        print(f"      Solusi:")
        print(f"      1. Kurangi skip_frames ke skip_frames=1")
        print(f"      2. Naikkan duration ke 20-30 detik")
        print(f"      3. Cek resolusi kamera (coba lower resolution)")
    elif actual_fps < 25:
        print(f"   ⚠️  FPS sedang ({actual_fps:.1f}fps)")
        print(f"      Untuk 10 detik: {int(actual_fps * 10)} frame")
        print(f"      Rekomendasi: Gunakan skip_frames=1, duration=15 detik")
    else:
        print(f"   ✓ FPS bagus ({actual_fps:.1f}fps)")
        print(f"      Untuk 10 detik: {int(actual_fps * 10)} frame")
    
    print(f"\n{'='*70}\n")
    
    return actual_fps, frame_count, actual_time

# Jalankan diagnosa
actual_fps, frame_count, actual_time = diagnose_camera_settings()



🔍 CAMERA DIAGNOSTIC

📹 Kamera Properties (dari driver):
   - FPS: 30.0
   - Resolution: 640x480
   - Frame count property: -1

⏱️  Testing actual FPS (recording 5 seconds)...
   ✓ Frames captured: 45
   ✓ Time elapsed: 5.07s
   ✓ Actual FPS: 8.87

📊 ANALISIS:
   ⚠️  FPS rendah! Hanya 8.9fps
      Untuk 10 detik: 88 frame (seharusnya ~300 frame)
      Solusi:
      1. Kurangi skip_frames ke skip_frames=1
      2. Naikkan duration ke 20-30 detik
      3. Cek resolusi kamera (coba lower resolution)




In [130]:
# ====== OPTIMIZED: CAPTURE DENGAN KONTROL FPS & DURATION ======
def capture_video_optimized(output_dir, duration_seconds=10, target_fps=30, 
                           resolution=(640, 480), speaker_name="ada"):
    """
    Capture video dengan optimasi untuk mendapatkan jumlah frame yang tepat
    
    target_fps: 30 (default), atau 15, 20 kalau camera lambat
    resolution: (640, 480) atau coba (320, 240) kalau ada issue
    """
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ ERROR: Kamera tidak bisa dibuka!")
        return None, 0, 0
    
    # 1. SET TARGET RESOLUTION (untuk optimize performance)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, resolution[0])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, resolution[1])
    cap.set(cv2.CAP_PROP_FPS, target_fps)
    
    # 2. GET ACTUAL SETTINGS
    actual_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"\n🎥 CAPTURE SETTINGS:")
    print(f"   Target FPS: {target_fps}")
    print(f"   Actual FPS: {actual_fps}")
    print(f"   Resolution: {width}x{height}")
    print(f"   Duration: {duration_seconds}s")
    print(f"   Expected frames: {int(actual_fps * duration_seconds)}")
    
    # 3. SETUP VIDEO WRITER
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(output_dir, f"{speaker_name}_{timestamp}.avi")
    
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(output_path, fourcc, int(actual_fps), (width, height))
    
    if not out.isOpened():
        print(f"❌ ERROR: Video writer gagal!")
        cap.release()
        return None, 0, 0
    
    # 4. RECORD
    print(f"\n🎥 Recording... (press 'q' untuk stop)")
    print(f"   Jumlah frame saat ini: ", end='', flush=True)
    
    start_time = time.time()
    frame_count = 0
    landmark_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print(" | ERROR reading frame")
            break
        
        # Detect landmarks
        mouth_landmarks, mouth_bbox, face_bbox = extract_mouth_landmarks(frame)
        if mouth_landmarks is not None:
            frame = draw_mouth_on_frame(frame, mouth_landmarks, mouth_bbox, face_bbox)
            landmark_count += 1
        
        # Write frame
        try:
            out.write(frame)
        except Exception as e:
            print(f" | ERROR: {e}")
            break
        
        elapsed = time.time() - start_time
        status = "✓" if mouth_landmarks is not None else "⚠"
        
        # Print progress setiap 30 frame
        if frame_count % 30 == 0:
            print(f"\r   Jumlah frame saat ini: {frame_count} | Elapsed: {elapsed:.1f}s | {status}", 
                  end='', flush=True)
        
        frame_count += 1
        
        # Stop condition
        if elapsed > duration_seconds:
            break
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n   Stopped by user")
            break
    
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    
    elapsed_total = time.time() - start_time
    
    # Statistics
    print(f"\n✅ RECORDING COMPLETE:")
    print(f"   Total frames: {frame_count}")
    print(f"   Actual time: {elapsed_total:.2f}s")
    print(f"   Actual FPS: {frame_count/elapsed_total:.2f}")
    
    if frame_count > 0:
        det_rate = 100 * landmark_count / frame_count
        print(f"   Landmarks detected: {landmark_count}/{frame_count} ({det_rate:.1f}%)")
    
    print(f"   Save to: {output_path}")
    
    return output_path, frame_count, int(actual_fps)

# CONTOH: Gunakan capture_optimized untuk testing
print("\n💡 GUNAKAN UNTUK DATA COLLECTION:")
print("""
# Test dengan optimized function - 20detik untuk 200 frame stabil
video_path, frames, fps = capture_video_optimized(
    output_dir='datasets/test_optimized',
    duration_seconds=20,
    target_fps=10,            # MediaPipe actual FPS
    resolution=(640, 480),
    speaker_name='test'
)

# Atau kalau butuh lebih banyak frame:
video_path, frames, fps = capture_video_optimized(
    output_dir='datasets/test_optimized',
    duration_seconds=30,      # Naikkan untuk 300 frame
    target_fps=10,            # MediaPipe @ 10fps
    resolution=(640, 480),
    speaker_name='test'
)
""")



💡 GUNAKAN UNTUK DATA COLLECTION:

# Test dengan optimized function - 20detik untuk 200 frame stabil
video_path, frames, fps = capture_video_optimized(
    output_dir='datasets/test_optimized',
    duration_seconds=20,
    target_fps=10,            # MediaPipe actual FPS
    resolution=(640, 480),
    speaker_name='test'
)

# Atau kalau butuh lebih banyak frame:
video_path, frames, fps = capture_video_optimized(
    output_dir='datasets/test_optimized',
    duration_seconds=30,      # Naikkan untuk 300 frame
    target_fps=10,            # MediaPipe @ 10fps
    resolution=(640, 480),
    speaker_name='test'
)



## 🔴 MASALAH: Hanya 96 Frame Untuk 10 Detik

**Root Cause**: Kamera hanya capture ~9.6 fps, bukan 30fps!
- 96 frame ÷ 10 detik = **9.6 fps** (seharusnya 30fps)
- Expected: 300 frame ✓
- Actual: 96 frame ✗

### ✅ SOLUSI CEPAT

| Masalah | Solusi | Code |
|---------|--------|------|
| FPS terlalu rendah (<15fps) | Naikkan durasi | `duration_seconds=20` |
| FPS sedang (15-25fps) | Kombinasi keduanya | `duration_seconds=15, target_fps=20` |
| Resolution tinggi → FPS rendah | Turunkan resolusi | `resolution=(320, 240)` |
| Perlu ==300 frame pasti | Skip frames 0 | `skip_frames=1` di extraction |

### 🎯 REKOMENDASI SETTING

```python
# ✓ OPTION 1: Kamera normal (30fps) - Gunakan ini dulu
capture_video_optimized(
    duration_seconds=10,
    target_fps=30,
    resolution=(640, 480)
)

# ✓ OPTION 2: Kamera slow (9-15fps) - Naikkan durasi
capture_video_optimized( 
    duration_seconds=20,        # 2x lebih lama
    target_fps=15,
    resolution=(640, 480)       # Atau (320,240) kalau masih lambat
)

# ✓ OPTION 3: Kamera sangat lambat - Kombinasi semua
capture_video_optimized(
    duration_seconds=30,        # 3x lebih lama
    target_fps=15,              # Target lebih rendah
    resolution=(320, 240)       # Resolusi lebih kecil
)
```

### 📊 EXPECTED RESULTS

| Setting | Expected Frame |
|---------|---|
| 10s @ 30fps, skip=1 | 300 |
| 20s @ 15fps, skip=1 | 300 |  
| 15s @ 20fps, skip=1 | 300 |
| 10s @ 10fps, skip=1 | 100 |

### 🔧 STEP-BY-STEP

1. **Jalankan diagnostic** untuk tahu FPS sebenarnya
2. **Hitung durasi yang butuh**: `durasi_butuh = 300 / actual_fps`
3. **Gunakan `capture_video_optimized()`** dengan durasi yang tepat
4. **Extract frames dengan `skip_frames=1`** untuk dapat semua frame


In [131]:
# ====== SOLUSI: MEDIAPIPE SLOW (10fps) ======

print("\n" + "="*70)
print("🔴 MediaPipe @ 10fps = BERAT!")
print("="*70)
print("""
Masalah: MediaPipe inference memakan ~100ms per frame
- 10fps = hanya 100 frame per 10 detik (bukan 300!)
- Solusi butuh: NAIKKAN DURASI atau OPTIMIZE inference

OPSI 1: NAIKKAN DURASI (TERMUDAH)
  ✓ Untuk 300 frame @ 10fps → butuh 30 detik
  
  capture_video_optimized(
      duration_seconds=30,      # NAIKKAN 3X
      target_fps=10,            # Set ke actual FPS
      resolution=(640, 480)
  )

OPSI 2: EXTRACT DENGAN SKIP FRAMES
  ✓ Kalau durasi terbatas, skip fewer frames saat extraction
  
  extract_frames_from_video(
      video_path,
      output_dir,
      skip_frames=1            # Extract SEMUA frame
  )

OPSI 3: OPTIMIZE MEDIAPIPE (ADVANCED)
  ✓ Turunkan confidence threshold
  ✓ Disable hand tracking kalau gak butuh
  ✓ Use lite model daripada full model
  ✓ Resize frame sebelum inference
""")

print("\n💡 REKOMENDASI CEPAT:")
print("""
# Kalau MediaPipe = 10fps:
metadata = collect_lip_reading_data(
    kata='halo',
    speaker_name='fahri',
    duration=30,        # ← NAIKKAN dari 10 ke 30
    skip_frames=1
)

# Ini akan dapat:
# 30 detik × 10 fps = 300 frame ✓
""")
print("="*70 + "\n")



🔴 MediaPipe @ 10fps = BERAT!

Masalah: MediaPipe inference memakan ~100ms per frame
- 10fps = hanya 100 frame per 10 detik (bukan 300!)
- Solusi butuh: NAIKKAN DURASI atau OPTIMIZE inference

OPSI 1: NAIKKAN DURASI (TERMUDAH)
  ✓ Untuk 300 frame @ 10fps → butuh 30 detik

  capture_video_optimized(
      duration_seconds=30,      # NAIKKAN 3X
      target_fps=10,            # Set ke actual FPS
      resolution=(640, 480)
  )

OPSI 2: EXTRACT DENGAN SKIP FRAMES
  ✓ Kalau durasi terbatas, skip fewer frames saat extraction

  extract_frames_from_video(
      video_path,
      output_dir,
      skip_frames=1            # Extract SEMUA frame
  )

OPSI 3: OPTIMIZE MEDIAPIPE (ADVANCED)
  ✓ Turunkan confidence threshold
  ✓ Disable hand tracking kalau gak butuh
  ✓ Use lite model daripada full model
  ✓ Resize frame sebelum inference


💡 REKOMENDASI CEPAT:

# Kalau MediaPipe = 10fps:
metadata = collect_lip_reading_data(
    kata='halo',
    speaker_name='fahri',
    duration=30,        # ← NAI

In [132]:
# ====== OPTIMASI: MEDIAPIPE FASTER ======
import mediapipe as mp

def optimize_mediapipe_settings():
    """
    Optimize MediaPipe untuk naikin FPS
    """
    print(f"\n{'='*70}")
    print(f"⚡ MEDIAPIPE OPTIMIZATION TIPS")
    print(f"{'='*70}\n")
    
    tips = [
        ("1. RESIZE FRAME SEBELUM INFERENCE", [
            "  Mediapipe: resize ke 480p/360p (bukan 1080p)",
            "  Code: frame_resized = cv2.resize(frame, (640, 480))",
            "  Impact: ↑ 20-30% FPS"
        ]),
        
        ("2. TURUNKAN CONFIDENCE THRESHOLD", [
            "  Ganti default 0.5 → 0.3 (lebih cepat, kurang akurat)",
            "  Code: min_detection_confidence=0.3",
            "  Impact: ↑ 10-15% FPS"
        ]),
        
        ("3. DISABLE TRACKING KALAU TIDAK PERLU", [
            "  Jangan enable static_image_mode atau tracking",
            "  Code: static_image_mode=False (sudah default)",
            "  Impact: ↑ 5-10% FPS"
        ]),
        
        ("4. USE LITE MODEL (kalau ada)", [
            "  MediaPipe punya lite version yang lebih cepat",
            "  Code: model_selection=0 (lite) vs 1 (full)",
            "  Impact: ↑ 40-50% FPS (!)"
        ]),
        
        ("5. SKIP FRAME PROCESSING", [
            "  Process setiap 2-3 frame, interpolate yang lain",
            "  Misal: process=1, skip=2 → 1 out of 3 frame",
            "  Impact: ↑ 100-200% FPS (!)"
        ]),
    ]
    
    for title, details in tips:
        print(f"📌 {title}")
        for detail in details:
            print(detail)
        print()
    
    print(f"{'='*70}\n")

optimize_mediapipe_settings()

# ====== QUICK COMPARISON ======
print("📊 EXPECTED FPS IMPROVEMENT:\n")
comparison = [
    ("Current", "10 fps", "100 frame / 10s", "❌ Kurang"),
    ("+ Lite Model", "15-20 fps", "150-200 frame / 10s", "✓ Lumayan"),
    ("+ Frame Skip", "20-30 fps", "200-300 frame / 10s", "✓✓ Bagus"),
    ("+ Duration=20s", "10 fps x 20s", "200 frame / 20s", "✓✓ Optimal"),
    ("Kombinasi Semua", "25-30 fps*", "~300 frame / 10s", "✓✓✓ Terbaik"),
]

print("| Strategy | FPS | Frame Count | Status |")
print("|----------|-----|-----|--------|")
for strategy, fps, frames, status in comparison:
    print(f"| {strategy:<20} | {fps:<8} | {frames:<18} | {status} |")

print("\n* Approximate dengan semua optimization")



⚡ MEDIAPIPE OPTIMIZATION TIPS

📌 1. RESIZE FRAME SEBELUM INFERENCE
  Mediapipe: resize ke 480p/360p (bukan 1080p)
  Code: frame_resized = cv2.resize(frame, (640, 480))
  Impact: ↑ 20-30% FPS

📌 2. TURUNKAN CONFIDENCE THRESHOLD
  Ganti default 0.5 → 0.3 (lebih cepat, kurang akurat)
  Code: min_detection_confidence=0.3
  Impact: ↑ 10-15% FPS

📌 3. DISABLE TRACKING KALAU TIDAK PERLU
  Jangan enable static_image_mode atau tracking
  Code: static_image_mode=False (sudah default)
  Impact: ↑ 5-10% FPS

📌 4. USE LITE MODEL (kalau ada)
  MediaPipe punya lite version yang lebih cepat
  Code: model_selection=0 (lite) vs 1 (full)
  Impact: ↑ 40-50% FPS (!)

📌 5. SKIP FRAME PROCESSING
  Process setiap 2-3 frame, interpolate yang lain
  Misal: process=1, skip=2 → 1 out of 3 frame
  Impact: ↑ 100-200% FPS (!)


📊 EXPECTED FPS IMPROVEMENT:

| Strategy | FPS | Frame Count | Status |
|----------|-----|-----|--------|
| Current              | 10 fps   | 100 frame / 10s    | ❌ Kurang |
| + Lite Model   

In [133]:
print("\n" + "="*70)
print("🎯 3 PILIHAN SOLUSI (PILIH 1)")
print("="*70)

print("""
═══════════════════════════════════════════════════════════════════════════

✅ PILIHAN 1: SIMPLE (Naikkan durasi aja)
─────────────────────────────────────────────────────────────────────────
• Durasi: 30 detik (dari 10)
• Expected: 300 frame (10fps × 30s)
• Pro: Sangat mudah, gak perlu ubah code
• Con: Recording lebih lama

CODE:
    metadata = collect_lip_reading_data(
        kata='halo',
        speaker_name='fahri',
        duration=30,        # ← Ubah dari 10 ke 30
        skip_frames=1
    )

═══════════════════════════════════════════════════════════════════════════

✅ PILIHAN 2: MEDIUM (Naikkan durasi+skip frame optimalkan)
─────────────────────────────────────────────────────────────────────────
• Durasi: 20 detik
• Skip frames saat extraction: 1-2 (extract semua/every 2nd)
• Expected: 200 frame (bagus untuk training)
• Pro: Balance antara durasi & jumlah frame
• Con: Perlu adjust skip_frames saat extraction

CODE:
    metadata = collect_lip_reading_data(
        kata='halo',
        speaker_name='fahri',
        duration=20,        # Moderate
        skip_frames=1       # Extract semua
    )

═══════════════════════════════════════════════════════════════════════════

✅ PILIHAN 3: ADVANCED (Optimize MediaPipe config)
─────────────────────────────────────────────────────────────────────────
• Ubah MediaPipe settings (lite model, lower confidence)
• Durasi: 10-15 detik
• Expected: 20+ fps → 200-300 frame per 10-15s
• Pro: Recording lebih singkat, FPS meningkat
• Con: Akurasi MediaPipe bisa turun sedikit

Steps:
    1. Edit `extract_mouth_landmarks()` function
    2. Add resize() sebelum inference
    3. Turun confidence threshold ke 0.3-0.4
    4. Use lite model kalau ada

═══════════════════════════════════════════════════════════════════════════
""")

print("🚀 REKOMENDASI UNTUK KAMU:")
print("-" * 70)
print("Kalau KPL (gak punya waktu) → PILIHAN 1 (naikkan ke 30s)")
print("Kalau balanced           → PILIHAN 2 (20s + optimize)")
print("Kalau mau cepat & banyak → PILIHAN 3 (optimize MediaPipe)")
print("="*70 + "\n")

print("💬 Mana yang mau aku setup? Atau run mana dulu testing?")



🎯 3 PILIHAN SOLUSI (PILIH 1)

═══════════════════════════════════════════════════════════════════════════

✅ PILIHAN 1: SIMPLE (Naikkan durasi aja)
─────────────────────────────────────────────────────────────────────────
• Durasi: 30 detik (dari 10)
• Expected: 300 frame (10fps × 30s)
• Pro: Sangat mudah, gak perlu ubah code
• Con: Recording lebih lama

CODE:
    metadata = collect_lip_reading_data(
        kata='halo',
        speaker_name='fahri',
        duration=30,        # ← Ubah dari 10 ke 30
        skip_frames=1
    )

═══════════════════════════════════════════════════════════════════════════

✅ PILIHAN 2: MEDIUM (Naikkan durasi+skip frame optimalkan)
─────────────────────────────────────────────────────────────────────────
• Durasi: 20 detik
• Skip frames saat extraction: 1-2 (extract semua/every 2nd)
• Expected: 200 frame (bagus untuk training)
• Pro: Balance antara durasi & jumlah frame
• Con: Perlu adjust skip_frames saat extraction

CODE:
    metadata = collect_lip_rea

In [134]:
# ====== FINAL SETTING: 20 DETIK (200 FRAME) ======
print(f"\n{'='*70}")
print(f"✅ FINAL SETUP CONFIRMED")
print(f"{'='*70}\n")

print("""
🎯 MEDIAIPIPE @ 10fps → 20 DETIK = 200 FRAME

SUMMARY PERUBAHAN:
✓ Duration default: 10s → 20s (2x lipat)
✓ Expected frame: 100 → 200
✓ Optimal untuk: Training CV model
✓ Recording time: Still reasonable (~20 detik per sample)

READY TO COLLECT:
""")

print("# Option 1: Single sample")
print("""
metadata = collect_lip_reading_data(
    kata='halo',
    speaker_name='fahri'
    # duration=20 & skip_frames=1 adalah default
)
""")

print("\n# Option 2: Batch collection")
print("""
config = [
    {"kata": "halo", "speaker": "fahri", "count": 5},
    {"kata": "halo", "speaker": "budi", "count": 3},
    {"kata": "terima_kasih", "speaker": "fahri", "count": 5},
]
results = batch_collect_data(config)
stats = get_dataset_statistics()
""")

print(f"\n{'='*70}\n")
print("💬 Run cell ini untuk mulai collection:")
print("    metadata = collect_lip_reading_data(kata='halo', speaker_name='fahri')")



✅ FINAL SETUP CONFIRMED


🎯 MEDIAIPIPE @ 10fps → 20 DETIK = 200 FRAME

SUMMARY PERUBAHAN:
✓ Duration default: 10s → 20s (2x lipat)
✓ Expected frame: 100 → 200
✓ Optimal untuk: Training CV model
✓ Recording time: Still reasonable (~20 detik per sample)

READY TO COLLECT:

# Option 1: Single sample

metadata = collect_lip_reading_data(
    kata='halo',
    speaker_name='fahri'
    # duration=20 & skip_frames=1 adalah default
)


# Option 2: Batch collection

config = [
    {"kata": "halo", "speaker": "fahri", "count": 5},
    {"kata": "halo", "speaker": "budi", "count": 3},
    {"kata": "terima_kasih", "speaker": "fahri", "count": 5},
]
results = batch_collect_data(config)
stats = get_dataset_statistics()



💬 Run cell ini untuk mulai collection:
    metadata = collect_lip_reading_data(kata='halo', speaker_name='fahri')
